[Guide Index](README.md)

# 05.02 The StarLayer SHACL engine

`starlayer.shacl` is StarLayer's SHACL engine. 

The engine can be used in two ways: through the methods of the `StarShaclProcessor` and through various functions. Most actions of the engine can use both approaches.  The various SHACL notebooks primarily demonstrate the use of the processor.  

This notebook describes the various functions and methods of the engine and where each is discussed in the guide.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.

In [1]:
from starlayer.graph import Namespace
from starlayer.graph import StarLayerGraph
from starlayer.graph.config import set_default_prefixes
from starlayer.shacl import StarShaclProcessor, target_nodes, validate

# EX and SH are starlayer.graph Namespaces: their terms are pyoxigraph NamedNodes, used to look terms up in the results below
# (StarLayerGraphs, see section 2.1); the StarLayerGraph inputs are built by parsing Turtle.
EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")

# Every StarLayerGraph made after this has these prefixes, so results, queries and the report text use short names such as
# ex:alice even for a graph built in Python rather than parsed from Turtle (a prefix the Turtle declares itself still applies).
set_default_prefixes({
    "ex": "http://example.org/",
    "sh": "http://www.w3.org/ns/shacl#",
    "shnex": "http://www.w3.org/ns/shacl-node-expr#",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
})


def show(graph):
    """Print a graph as pretty Turtle, declaring only the prefixes it uses."""
    print(graph.serialize(format="turtle", pretty=True, only_used=True))

## 05.02.1 Index of functions and methods

For each operation of the engine, the table shows the function, whether it is also a method of a `StarShaclProcessor` ✅, what it does, and where it is discussed. A method uses the graphs and settings the processor holds. 

Everything in the table, and `StarShaclProcessor`, are imported from `starlayer.shacl`.

| Function | Method | What it does | Discussed in |
|---|:-:|---|---|
| **Validating, evaluating and running rules** | | | |
| `validate()` | ✅ | Checks a data graph against a shapes graph and returns a `ValidationReport`. | [05.01](05.01-validating-with-shacl-1.2.ipynb) |
| `validate_each()` | ✅ | Validates each graph of a dataset against the shapes graph. | [05.01.4.1](05.01-validating-with-shacl-1.2.ipynb) |
| `evaluate()` | ✅ | The values of a node or nodes for the properties of a shape, with `sh:values` calculated and `sh:defaultValue` used, as a graph. | [05h](05h-shacl-evaluate.ipynb) |
| `apply_rules()` | ✅ | Runs the shapes' rules over the data until nothing new is inferred. Returns the inferred triples. The data graph is not changed. | [05c](05c-shacl-inference-rules.ipynb) |
| `target_nodes()` | ✅ | The nodes a shape targets in the data graph, which are the nodes `validate()` checks for that shape. | [05.01.3.6](05.01-validating-with-shacl-1.2.ipynb), [05.03](05.03-shapes-as-classes.ipynb) |
| `target_shapes()` | ✅ | The shapes that target a node, the reverse of `target_nodes()`. | [05.01.3.6](05.01-validating-with-shacl-1.2.ipynb), [05.03](05.03-shapes-as-classes.ipynb) |
| **Writing and extracting shapes** | | | |
| `derive_shape()` |  | Builds one closed node shape from a class or a node in the data graph. Used to start designing a shape from data. | section 05.02.3 |
| `extract_subgraph()` | ✅ | The part of the data graph that one shape covers for a focus node, or that all the shapes of a shapes graph cover. | [05f](05f-shacl-subgraph-extraction.ipynb) |
| `close_shape()` | ✅ | A copy of the shapes graph in which a shape, and the shapes it reaches, are closed. | [05f](05f-shacl-subgraph-extraction.ipynb) |
| `extract_shapes_graph()` | ✅ | One named `sh:ShapesGraph` as a graph of its own, with its members and prefixes. | [05e](05e-shacl-profiling.ipynb) |
| **SHACL UI and editors** | | | |
| `select_widgets()` | ✅ | The widget for each property shape of a node shape, for a given role (editor or viewer). | [05d](05d-shacl-ui.ipynb) |
| `select_widget()` | ✅ | The widget for one property shape and given focus node, by the SHACL UI selection procedure. | [05d](05d-shacl-ui.ipynb) |
| `explain_widgets()` | ✅ | Every candidate widget for a property shape, and the steps that chose the selected widget. | [05d](05d-shacl-ui.ipynb) |
| `candidate_values()` | ✅ | What a form can offer for a field: the values the shape names, with the instances of a class and its subclasses. | [05g](05g-building-on-shacl.ipynb) |
| `value_form()` | ✅ | Whether a field holds one value, an RDF list, or either, read from its shape. | [05g](05g-building-on-shacl.ipynb) |
| `check_value()` | ✅ | Whether a value would be accepted as valid for a field of a node, before it is saved. | [05g](05g-building-on-shacl.ipynb) |
| `form_model()` | ✅ | The fields of a node's form, with their widgets, their values and source of each value. | [05g](05g-building-on-shacl.ipynb) |
| **Checking shapes** | | | |
| `meta_validate()` |  | Checks a shapes graph against SHACL's own shapes, or a proposed edit to a shapes graph, before it is made. | section 05.02.4 |
| `meta_scope()` |  | What a check of an edit would cover, and why. | section 05.02.4.2 |
| **Reading a report** | | | |
| `report_results()` |  | The results of a validation report. | [05g](05g-building-on-shacl.ipynb) |
| `diff_reports()` |  | Compares two validation reports and outputs the changes. | [05g](05g-building-on-shacl.ipynb) |
| `derive_conforms_to()` |  | Derives `sh:conformsToShapesGraph` from a validation report and an optional `specification_graph` | [05e](05e-shacl-profiling.ipynb) |


A function takes the graphs and optional parameters on every call. `StarShaclProcessor` binds a shapes graph, and optionally other graphs and settings, once, so that it can be re-used with different data graphs, and offers nearly all of the same operations as methods. 

The rest of this guide covers the functionality of the SHACL engine not covered elsewhere:

- how the processor is configured, and how a call overrides the configuration (section 05.02.2),
- how to derive a shape from data (section 05.02.3),
- how to check a shapes graph (section 05.02.4),
- the errors (section 05.02.5), and
- descriptive shape properties (section 05.02.6).

The functions and the processor take a `StarLayerGraph`, a `StarLayerDataset` or a `pyoxigraph.Store`. Anything else raises a `TypeError` that names the argument.

The results and errors are classes: `ValidationReport` (the result of `validate()`), `RulesResult` (the result of `apply_rules()`), and the errors `ShapeError`, `RuleError` and `MetaShaclError`, which are all `ValueError`s (see [05.01.4.2](05.01-validating-with-shacl-1.2.ipynb)). `SH` is a namespace of the SHACL terms.

## 05.02.2 Configuring the processor

A `StarShaclProcessor` is configured when it is created. Only the shapes graph is required: every other graph and every setting is optional. Configurations set at creation can be modified. Any configuration can be overridden during an individual method call without changing the configuration. From lowest to highest, the precedence is the built-in default, the processor's configuration, then the call. The functions have no configuration of their own: they take the same settings as keywords on each call.

The processor holds its graphs and does not copy them, so a change made to the data graph or the shapes graph is seen by the next method call. 

The graphs can be given when the processor is created.  Only the shapes graph is required.

| Graph | What it is |
|---|---|
| `shacl_graph` | the shapes graph |
| `ont_graph` | an ontology graph, read when an entailment regime is used ([05.01.4.3](05.01-validating-with-shacl-1.2.ipynb)) |
| `data_graph` | a default data graph, so that `validate()` or `evaluate()` can be called without one |
| `scoring_graph` | the widget scoring catalog for `select_widgets()` (see the [SHACL UI guide](05d-shacl-ui.ipynb)) |

The settings are keywords after the graphs. Each is held by the processor and used by the calls that need it. Related settings are together: checking the shapes graph and its imports, entailment and the class hierarchy, conformance, and limits:

| Setting | Built-in default | What it does | Discussed in |
|---|---|---|---|
| `meta_shacl` | `False` | check the shapes graph against SHACL's own SHACL for shapes when the processor is created, and raise `MetaShaclError` if it does not conform | section 05.02.4 |
| `meta_shapes_extra` | none | additional shapes to be included in the meta_shacl check | section 05.02.4.1 |
| `ignore_imports`, `shapes_graph_loader` | `False`, none | how `owl:imports` of the shapes graph are resolved. A declared import that is not in the graph raises an error unless `shapes_graph_loader` is a function that supplies it or `ignore_imports=True` | section 05.02.2.2 |
| `inference` | none | an entailment regime IRI such as `ENTAILMENT.RDFS`, or several, applied to the data graph of every call | [05.01.4.3](05.01-validating-with-shacl-1.2.ipynb) |
| `recompute` | `"onChange"` | when the entailed triples of `inference` are computed again: `"onChange"` (when the graph has changed), `"never"` or `"always"`, as for a SPARQL query | [05.01.4.3](05.01-validating-with-shacl-1.2.ipynb) |
| `engine`, `timeout` | `"hermit"`, none | for the `OWL-Direct` regime only: the reasoner and its time limit, as for a SPARQL query | [05.01.4.3](05.01-validating-with-shacl-1.2.ipynb) |
| `subclass_of_in_shapes_graph` | `False` | also read `rdfs:subClassOf` from the shapes graph | [05.03.2](05.03-shapes-as-classes.ipynb) |
| `allow_warnings`, `allow_infos` | `False` | a warning (or an info) no longer makes `conforms` false | [05.01.3.7](05.01-validating-with-shacl-1.2.ipynb) |
| `max_iterations` | 1000 | limit on the rounds of `apply_rules()`, the method and the function | [05c](05c-shacl-inference-rules.ipynb), section 12 |

### .2.1 Overriding the configuration

The configuration of a processor is the starting point of each call, not a fixed set of values. A setting has up to three values, and the highest of them is the one a call uses:

1. the built-in default of the setting (see the table above),
2. the processor's configuration: the value given when the processor was created, or set later by assigning the attribute (`myProcessor.allow_warnings = True`), which changes the configuration for every call after that, and
3. the call: a keyword passed to a method of the processor, such as `validate(allow_warnings=True)`, which applies to that call only and leaves the processor's configuration as it was.

The same keywords are used at each level. A function has no configuration, so only its built-in defaults and the keywords of the call apply. A method call can also provide a value the processor lacks, such as a data graph when none is bound. This makes a processor a reusable base: it is configured once for what is usual, and a call overrides it for exceptions.

Below, `ex:bob` has no name, which is a warning. With the built-in default, `allow_warnings=False`, a warning makes him not conform. Assigning `myProcessor.allow_warnings = True` changes the configuration for every call after that (conforms). A method call with `allow_warnings=False` overrides the processor's setting for that call only (does not conform), and the setting remains unchanged. The warning is in the validation results every time: `allow_warnings` changes only whether it makes `conforms` false.

In [2]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ; sh:severity sh:Warning ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:bob a ex:Person .
""", format="turtle")


def show_results(label, report):
    results = [f"{r.severity_name}: {r.messages[0].value}" for r in report.results]
    print(f"{label}: conforms={report.conforms}, results={results}")


# 1. the built-in default: allow_warnings is False, so a warning makes conforms false
myProcessor = StarShaclProcessor(shacl_graph=shapes, data_graph=data)
show_results("built-in default", myProcessor.validate())

# 2. the processor's configuration: assigning the attribute changes it for every call after that
myProcessor.allow_warnings = True
show_results("the processor's setting", myProcessor.validate())

# 3. a call overrides the setting for that call only
show_results("the call's setting", myProcessor.validate(allow_warnings=False))
print("the processor's setting after the call:", myProcessor.allow_warnings)

built-in default: conforms=False, results=['Warning: Needs at least 1 value for ex:name']
the processor's setting: conforms=True, results=['Warning: Needs at least 1 value for ex:name']
the call's setting: conforms=False, results=['Warning: Needs at least 1 value for ex:name']
the processor's setting after the call: True


### .2.2 Imports of the shapes graph: `ignore_imports` and `shapes_graph_loader`

A shapes graph can import another with `owl:imports`. The SHACL spec says a processor should follow the imports before it validates. An import that is not in the shapes graph raises an error at the first `validate()`. Skipping and import would drop the shapes it supplies, any validation of a data graph would be incomplete.  

There are three ways to handle an import that is not already in the shapes graph:

- **Manually merge the imported graph into the shapes graph.** An import counts as present when its IRI is the subject of triples in the shapes graph.
- **Give a `shapes_graph_loader`.** It is a function that is called with the IRI of the import, a `pyoxigraph.NamedNode`, and returns a graph, or `None` when it has none. The imported shapes then take part as if they were in the shapes graph.
- **Set `ignore_imports=True`.** The imports are not followed, and the shapes they would have supplied are not used. It is for imports that need no loading: the modules are already merged into the shapes graph but the `owl:imports` statements were kept, or the import is only a declaration.

The import of the SHACL vocabulary, `owl:imports sh:` do not need an import because the engine has the vocabulary built in. 

The [SHACL profiling guide](05e-shacl-profiling.ipynb) covers the packaging conventions for `owl:imports` and `sh:ShapesGraph`.

Below, `ex:PersonShape` requires every person to conform to `ex:CommonShape`, which is in a graph that the shapes graph imports. Without a loader, validating raises an error. With a loader that returns the common shapes, `ex:alice`, who has no name, does not conform. With `ignore_imports=True` the common shapes are not used, so she conforms.

In [3]:
main = StarLayerGraph()
main.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Main a owl:Ontology ; owl:imports <http://example.org/common> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ; sh:node ex:CommonShape .
""", format="turtle")

common = StarLayerGraph()
common.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:CommonShape a sh:NodeShape ; sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

people = StarLayerGraph()
people.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
""", format="turtle")

# the import is not in the shapes graph, and nothing supplies it
try:
    StarShaclProcessor(shacl_graph=main).validate(data_graph=people)
except ValueError as error:
    print("no loader:", error)

# a loader supplies the imported graph for its IRI
def loader(iri):
    return common if iri.value == "http://example.org/common" else None

print("with a loader, conforms:", StarShaclProcessor(shacl_graph=main, shapes_graph_loader=loader).validate(data_graph=people).conforms)

# the imports are not followed
print("with ignore_imports, conforms:", StarShaclProcessor(shacl_graph=main, ignore_imports=True).validate(data_graph=people).conforms)

no loader: shacl_graph declares owl:imports ['http://example.org/common'] that aren't already locally satisfied, but no shapes_graph_loader was given to resolve them. Pass shapes_graph_loader=... to resolve them, or ignore_imports=True if you've already resolved them yourself (or don't need them).
with a loader, conforms: False
with ignore_imports, conforms: True


## 05.02.3 Deriving shapes from data

`derive_shape()` is a function of `starlayer.shacl`, with no method form.  

Given a data graph and either a class or an individual node, it creates a shapes graph with one node shape targeting the class or node.  With `cls=`, the shape targets all instances of the class, subclasses included, and the shape and its constraints come from the properties of those class instances.   With `node=`, the shape targets that one node and the shape and its constraints come from the properties of that node.   

The shape is named after its target (`ex:Person` gives `ex:PersonShape`, `ex:alice` gives `ex:AliceShape`) unless `shape=` provides a name. From the values it finds, it sets the constraints the data supports: `sh:minCount` and `sh:maxCount`, the kind of value (`sh:datatype`, `sh:class` or `sh:nodeKind`) and numeric and string ranges. The shape is `sh:closed`, with `rdf:type` and the usual descriptive properties ignored. It is validated against the same data graph that was used to create it.

**Why use it.** A shape written from scratch can miss how the data is really used. `derive_shape()` gives a starting point drawn from the data itself, which is useful in three situations:

- **There are no shapes yet.** The derived shape shows what the data contains: which properties, how many values, which kinds of value. It can be read, edited and kept as the first version of the shape.
- **Later data should look like earlier data.** A shape derived from data that is known to be good can be used to validate data that arrives later, so that a change in its structure shows up as a violation.
- **The shapes should follow the data.** An existing shape can be given back as `template_graph`, and `derive_shape()` brings it up to date with the data, keeping what still holds (section 05.02.3.1).

A derived shape describes the data it was derived from, not what is intended. For example, `sh:maxCount 1` may only mean that the sample had one value. Constraints should be reviewed before relying on them as rules.

Below, a shape is derived for the class `ex:Person` and another for the node `ex:alice`, and the data is validated against the class shape (conforms).

In [4]:
import warnings

from starlayer.shacl import derive_shape

observed = StarLayerGraph()
observed.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:name "Alice" ; ex:age 30 ; ex:knows ex:bob .
    ex:bob a ex:Person ; ex:name "Bob" ; ex:age 41 .
""", format="turtle")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # the fidelity warning does not matter here
    # a shape for all the instances of a class
    class_shape = derive_shape(observed, cls=EX.Person)
    # a shape for one node
    node_shape = derive_shape(observed, node=EX.alice)
    # the data conforms to the shape derived from it
    conforms = StarShaclProcessor(shacl_graph=class_shape).validate(data_graph=observed).conforms
print("a shape for the class ex:Person")
show(class_shape)
print("a shape for the node ex:alice")
show(node_shape)
print("the data conforms to its own derived shape:", conforms)


a shape for the class ex:Person
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:PersonShape a sh:NodeShape ;
    sh:closed true ;
    sh:ignoredProperties (
        <http://purl.org/dc/terms/created>
        <http://purl.org/dc/terms/modified>
        rdf:type
        rdfs:comment
        rdfs:label
    ) ;
    sh:property [
        a sh:PropertyShape ;
        sh:class ex:Person ;
        sh:maxCount 1 ;
        sh:path ex:knows
    ], [
        a sh:PropertyShape ;
        sh:datatype xsd:integer ;
        sh:maxCount 1 ;
        sh:minCount 1 ;
        sh:path ex:age
    ], [
        a sh:PropertyShape ;
        sh:datatype xsd:string ;
        sh:maxCount 1 ;
        sh:minCount 1 ;
        sh:path ex:name
    ] ;
    sh:targetClass ex:Person .

a shape for the node ex:alice
@prefi

### .3.1 Updating a shape: `template_graph`

`derive_shape()` can start from an existing shape and bring it up to date with the data. It is given three things:

| What | How it is given |
|---|---|
| the data the shape is derived from | the first argument, as before |
| what the shape is about | `cls=` or `node=`, as before |
| the earlier version of the shape | `template_graph=`, a shapes graph that holds it |

`template_graph` is a graph, not a node: you do not point at the shape in it. `derive_shape()` looks in `template_graph` for the node shape with the same target: `sh:targetClass` for `cls=`, `sh:targetNode` for `node=`. If there is no such shape, or more than one, it raises a `ValueError`; with more than one, `shape=` says which to use. Only that shape and what it references (its property shapes and lists) are used, and the other shapes in it are not copied.

The result is a new shapes graph with the updated shape. Neither `template_graph` nor the data graph is changed. The shape keeps the name it had in the template, unless `shape=` gives another.

What happens to each constraint of the template shape depends on the data:

| The template's constraint | The result |
|---|---|
| holds for the data | stays as it is. Everything the data cannot decide stays too: `sh:name`, `sh:message`, and the rest |
| is broken by the data, and can be worked out again: counts, datatype, class, node kind, ranges, lengths, languages, `sh:in` | is recomputed from the data (`sh:in` is widened to include the new values) |
| is broken by the data, and cannot be worked out again: `sh:pattern`, `sh:hasValue`, comparisons between two properties | is dropped, with an `rdfs:comment` that says why |
| cannot be checked against the data: `sh:sparql` and custom constraint components | stays, with a comment |
| a property the data has and the template lacks | is added |

The template's `sh:ignoredProperties` are kept, and only the main shape is closed, not the shapes it merely references.

Below, an existing `ex:PersonShape` in `template_graph` is brought up to date with the graph `later`, which holds newer data about people. Alice now has two names, so `sh:maxCount 1` on `ex:name` is recomputed as 2, and the `sh:name "Name"` the person wrote stays. Her nickname `Al_1` breaks the `sh:pattern`, which cannot be worked out again, so it is dropped with a comment. `ex:age` still holds and is untouched, and `ex:email` is a new property, so it is added.

In [5]:
template_graph = StarLayerGraph()
template_graph.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ; sh:closed true ;
      sh:property [ sh:path ex:name ; sh:datatype xsd:string ; sh:minCount 1 ; sh:maxCount 1 ; sh:name "Name" ] ,
                  [ sh:path ex:age ; sh:datatype xsd:integer ; sh:maxCount 1 ] ,
                  [ sh:path ex:nick ; sh:pattern "^[a-z]+$" ] .
""", format="turtle")

later = StarLayerGraph()
later.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:name "Alice", "Ally" ; ex:age 30 ; ex:nick "Al_1" ; ex:email "alice@example.org" .
    ex:bob a ex:Person ; ex:name "Bob" ; ex:age 41 .
""", format="turtle")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # the fidelity warning does not matter here
    # the data graph, what the shape is about, and the graph that holds the earlier shape
    updated = derive_shape(later, cls=EX.Person, template_graph=template_graph)
show(updated)

@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:PersonShape a sh:NodeShape ;
    sh:closed true ;
    sh:ignoredProperties (
        <http://purl.org/dc/terms/created>
        <http://purl.org/dc/terms/modified>
        rdf:type
        rdfs:comment
        rdfs:label
    ) ;
    sh:property [
        a sh:PropertyShape ;
        sh:datatype xsd:string ;
        sh:maxCount 1 ;
        sh:path ex:email
    ], [
        sh:datatype xsd:integer ;
        sh:maxCount 1 ;
        sh:path ex:age
    ], [
        rdfs:comment "derive_shape(): PatternConstraintComponent no longer holds for the current data and has no principled auto-fix - dropped." ;
        sh:path ex:nick
    ], [
        sh:datatype xsd:string ;
        sh:maxCount 2 ;
        sh:minCount 1 ;
        sh:name "Name" ;
     

## 05.02.4 Checking a shapes graph: `meta_validate()`

`meta_validate()` checks a shapes graph against SHACL's own shapes to test whether it is well formed. It is a function and needs no processor, so it can check shapes before they are saved or used.

`meta_validate()` is an ordinary SHACL validation. The shapes graph is the data graph, and the shapes are `shacl_shacl`, the SHACL 1.2 shapes for shapes (`starontology.get_ontology_graph("shacl_shacl")`, from `starlayer/starontology/shacl/shacl12-shacl.ttl`). The result is a standard validation report, with the same parts as the report of `validate()`: `conforms`, the `report_graph` and the `report_text`. Reading a validation report, its results and its severities, is covered in [05.01.4](05.01-validating-with-shacl-1.2.ipynb).

- A valid shapes graph returns `(conforms, report_graph, report_text)`, with `conforms` true.
- A shapes graph that is not valid raises a `MetaShaclError`, which carries the report, with `conforms` false, in `report_graph` and `report_text`. The error also has a `problems` list, one line for each result in the report that describes the issue in words, and a `scope`, which says what the check covered (section 05.02.4.2).

A processor created with `meta_shacl=True` makes the same check when it is created, and raises the same `MetaShaclError`. That is not often needed, because shapes that are finished do not have to be checked each time they are used. It is useful when the processor is used while the shapes themselves are being edited.

Below, a valid shapes graph passes, and a shapes graph with two `sh:datatype` values on one property shape raises a `MetaShaclError`. The error holds the problem and the report.

In [10]:
from starlayer.shacl import MetaShaclError, meta_validate

valid = StarLayerGraph()
valid.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")
conforms, report_graph, report_text = meta_validate(valid)
print("valid shapes:", conforms)
print()

two_datatypes = StarLayerGraph()
two_datatypes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:datatype xsd:string, xsd:integer ] .
""", format="turtle")
try:
    meta_validate(two_datatypes)
except MetaShaclError as error:
    print(error.report_text)
    print("problems:", error.problems)

valid shapes: True

Validation Report
Conforms: False
Results (1):
Violation in MaxCountConstraintComponent (e37ec35afb3773e4650ff1873337bb4c):
	Focus Node: e054322ea55dd6bf3f840582a64b9057
	Message: Allows at most 1 value, found 2
problems: ('the property shape of ex:PersonShape for ex:name, sh:datatype: Allows at most 1 value, found 2',)


### .4.1 Adding your own checks: `extra_graphs`

`meta_validate()` takes `extra_graphs`, additional shapes to check the shapes graph against, along with SHACL's own. They are ordinary shapes whose data is the shapes graph, so a rule about how shapes are written is a shape that targets `sh:NodeShape` or `sh:PropertyShape`. This is a way to apply a house style, such as requiring every node shape to have a label, when shapes are created. A processor created with `meta_shacl=True` takes the same graphs as `meta_shapes_extra`.

Below, a shape requires every node shape to have an `rdfs:label`. A shapes graph without labels passes SHACL's own check but fails this one, and one with a label passes.

In [ ]:
extra = StarLayerGraph()
extra.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:LabelRule a sh:NodeShape ;
      sh:targetClass sh:NodeShape ;
      sh:property [ sh:path rdfs:label ; sh:minCount 1 ; sh:message "A node shape needs an rdfs:label" ] .
""", format="turtle")

unlabelled = StarLayerGraph()
unlabelled.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person .
""", format="turtle")

labelled = StarLayerGraph()
labelled.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:PersonShape a sh:NodeShape ; rdfs:label "Person" ; sh:targetClass ex:Person .
""", format="turtle")

meta_validate(unlabelled)  # SHACL's own shapes accept it
print("SHACL's own shapes accept the shapes graph without a label")
try:
    meta_validate(unlabelled, extra_graphs=[extra])
except MetaShaclError as error:
    print("with the extra shape:", error.problems)
meta_validate(labelled, extra_graphs=[extra])
print("with a label: accepted")

SHACL's own shapes accept the shapes graph without a label
with the extra shape: ('ex:PersonShape, rdfs:label: A node shape needs an rdfs:label',)
with a label: accepted


### .4.2 Checking an edit before it is made

`meta_validate(shapes_graph, add=..., remove=...)` takes the triples of a proposed edit and checks the shapes graph as it would be after them. The graph itself is not changed.

- Only the nodes the edit touches are checked, together with the shapes that refer to them.
- Every other node stays visible to the check, so a shape's type, the prefixes of a query and the parameters of a constraint component are still read from the whole graph.
- On a large shapes graph that takes a fraction of a second, where the full check takes seconds. `full=True` checks the whole graph, whatever the edit is.

Some edits can change what other nodes mean: a prefix declaration, a constraint component with its parameters and validators, a rule template, and the type of a node that already exists. These are risky, and the check then covers the whole graph. A new node, with the shapes and properties that come with it, is not risky.

`meta_scope()` gives the decision without running the check: a `MetaScope` with `full` (true when the whole graph is checked), the `nodes` that are covered, and the `reasons`. A `MetaShaclError` carries the scope it used in `scope`. A check of part of the graph is for the moment between edits: run the full check before the shapes are saved or published.

Below, an edit that adds a second `sh:datatype` is refused before it is made, and the graph is unchanged. `meta_scope()` shows that this edit is checked on its own nodes, while an edit to a prefix declaration needs the whole graph. A harmless edit with `full=True` is checked on the whole graph.

In [ ]:
from starlayer.graph import Literal, Namespace, StarLayerGraph
from starlayer.shacl import SH, MetaShaclError, meta_scope, meta_validate

EX = Namespace("http://example.org/")
XSD = Namespace("http://www.w3.org/2001/XMLSchema#")
current = StarLayerGraph()
current.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ; sh:property ex:AgeShape .
    ex:AgeShape sh:path ex:age ; sh:datatype xsd:integer .
""", format="turtle")

second_datatype = [(EX.AgeShape, SH.datatype, XSD.string)]  # an edit not yet made
try:
    meta_validate(current, add=second_datatype)
except MetaShaclError as error:
    print("edit refused:", error.scope.reasons)
print("the graph is unchanged:", len(list(current.triples((EX.AgeShape, SH.datatype, None)))), "datatype")

prefix_edit = [(EX.PersonShape, SH.declare, EX.Prefixes)]
print("full check for the datatype edit:", meta_scope(current, add=second_datatype).full)
print("a prefix edit:", meta_scope(current, add=prefix_edit).reasons)

name_edit = [(EX.AgeShape, SH.name, Literal("Age"))]  # a harmless edit
print("full=True forces the whole graph:", meta_scope(current, add=name_edit, full=True).full)
conforms, _report_graph, _report_text = meta_validate(current, add=name_edit, full=True)
print("the harmless edit, checked in full:", conforms)

edit refused: ('checked 3 nodes: 2 changed, 1 dependent',)
the graph is unchanged: 1 datatype
full check for the datatype edit: False
a prefix edit: ('full check: sh:declare changed on ex:PersonShape',)
full=True forces the whole graph: True
the harmless edit, checked in full: True


## 05.02.5 Errors

A SHACL engine raises one of four errors of its own, and a `TypeError` for a call it cannot take. Three of the four are `ValueError`s, so one `except ValueError` catches them. `EvaluationFailure` is a plain `Exception`. Each message says which shape, property, rule or argument it is about.

| Error | Raised when | It carries | Discussed in |
|---|---|---|---|
| `MetaShaclError` | The shapes graph does not conform to SHACL's own shapes when validated with `meta_validate()`. | A standard shacl validation report, along with a set of human-readable `problems` and `scope` of the validation. | section 05.02.4 |
| `ShapeError` | The engine cannot read or compile the shapes graph: a value of the wrong kind such as `sh:minCount "many"`, a construct SHACL 1.2 dropped, an invalid regular expression, or an `owl:imports` that cannot be resolved. Raised when the processor is created. A mistake that only shows when a query or expression runs, such as the text of a SPARQL query, is raised by the first `validate()`. | A message that names the shape and the property. | [05.01.4.2](05.01-validating-with-shacl-1.2.ipynb) |
| `RuleError` | A rule fails, the rules graph is not well formed, or `max_iterations` is exceeded. The rule set fails as a whole. | A message that names the rule. | [05c](05c-shacl-inference-rules.ipynb) |
| `EvaluationFailure` | A node expression cannot be evaluated, which is an error and not an empty result: for example a SPARQL function given arguments it cannot take, such as a division by zero. `evaluate_expression()` raises it. | The expression that failed. | [05g](05g-building-on-shacl.ipynb) |
| `TypeError` | A call is given something it cannot take. It is Python's own error and a mistake in the calling code, not a SHACL error: a graph of the wrong kind (an rdflib graph has to be converted with `StarLayerGraph.from_rdflib()`), `validate_each()` given anything but a dataset, or an option the function does not take, such as `allow_warnings` on `apply_rules()`. | A message that names the argument. | The example below |

**`ShapeError` and `MetaShaclError`.** Both are about a shapes graph that is not valid, and they overlap. A `ShapeError` comes from the engine itself, which stops at the first thing it cannot turn into a check. A `MetaShaclError` comes from validating the shapes graph against SHACL's own shapes: it reports every problem at once, and it also finds shapes the engine can read and would run, such as two `sh:datatype` values on one shape. The meta check is optional, and when it is on it runs first, so a mistake that would otherwise be a `ShapeError` is reported as a `MetaShaclError`. The same `sh:minCount "many"` raises a `ShapeError` by default and a `MetaShaclError` with `meta_shacl=True`.

**`EvaluationFailure`.** Evaluating a node expression for a focus node has one of two outcomes. It gives a list of nodes, which may be empty: an empty list is a normal answer and means no values. Or it fails: the expression could not be computed, as when a SPARQL function is given arguments it cannot take, such as `sparql:divide` by zero or `sparql:abs` of a string. `evaluate_expression()` returns the list or raises `EvaluationFailure`, so that the caller can tell the two apart. Where SHACL says what a failure means, the engine applies that itself.

Below, a shape with `sh:minCount "many"` raises a `ShapeError` when the processor is created, and a `MetaShaclError` with `meta_shacl=True`. A shape with two `sh:datatype` values, which the engine can read, raises a `MetaShaclError` from `meta_validate()`. A division by zero in a node expression raises an `EvaluationFailure`, and an option that `apply_rules()` does not take raises a `TypeError`.

In [9]:
from starlayer.graph import Namespace, StarLayerGraph
from starlayer.shacl import EvaluationFailure, MetaShaclError, ShapeError, StarShaclProcessor, apply_rules, evaluate_expression, meta_validate

EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")
XSD = Namespace("http://www.w3.org/2001/XMLSchema#")

def shapes(turtle):
    graph = StarLayerGraph()
    graph.parse(data="@prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .\n" + turtle, format="turtle")
    return graph

def first_line(error):
    return str(error).splitlines()[0]

# a shape the engine cannot read: raised when the processor is created
unreadable = shapes('ex:S a sh:NodeShape ; sh:property [ sh:path ex:p ; sh:minCount "many" ] .')
try:
    StarShaclProcessor(shacl_graph=unreadable)
except ShapeError as error:
    print("ShapeError:", first_line(error))

# a shape the engine can read but SHACL's own shapes reject
two_datatypes = shapes('ex:S a sh:NodeShape ; sh:property [ sh:path ex:p ; sh:datatype xsd:string, xsd:integer ] .')
try:
    meta_validate(two_datatypes)
except MetaShaclError as error:
    print("MetaShaclError:", len(error.problems), "problem:", error.problems[0])

# the shapes graph the engine cannot read, with the meta check on: SHACL's own shapes reject it first
try:
    StarShaclProcessor(shacl_graph=unreadable, meta_shacl=True)
except MetaShaclError as error:
    print("with meta_shacl=True, MetaShaclError:", error.problems[0])

# the first three are ValueErrors, so one except clause covers them
for check in (lambda: StarShaclProcessor(shacl_graph=unreadable), lambda: meta_validate(two_datatypes)):
    try:
        check()
    except ValueError as error:
        print("caught as ValueError:", type(error).__name__)

# not a SHACL error: an option the function does not take
try:
    apply_rules(StarLayerGraph(), shapes("ex:S a sh:NodeShape ."), allow_warnings=False)
except TypeError as error:
    print("TypeError:", first_line(error).split(":")[0])

# a node expression with no result: a SPARQL function given arguments it cannot take
holder = shapes("ex:holder ex:expression [ <http://www.w3.org/ns/sparql#divide> ( 1 0 ) ] .")
expression = next(holder.objects(EX.holder, EX.expression))
try:
    evaluate_expression(holder, StarLayerGraph(), expression, EX.focus)
except EvaluationFailure as error:
    print("EvaluationFailure:", error)

ShapeError: the property shape of http://example.org/S for http://example.org/p: sh:minCount needs exactly one non-negative xsd:integer, got ['many']
MetaShaclError: 1 problem: the property shape of ex:S for ex:p, sh:datatype: Allows at most 1 value, found 2
with meta_shacl=True, MetaShaclError: the property shape of ex:S for ex:p, sh:minCount: Value "many" should have the datatype xsd:integer
caught as ValueError: ShapeError
caught as ValueError: MetaShaclError
TypeError: apply_rules() does not take allow_warnings
EvaluationFailure: SPARQL error in (?a0 / ?a1)


## 05.02.6 Descriptive shape properties

Shapes can contain descriptive shape properties that describe a shape and have no impact on validation or other operations. People and tools can read the properties, but a validating processor should ignore them. The descriptive properties of SHACL are outlined below.  

- `sh:name` and `sh:description` name and describe what the shape is about: the property of a property shape, or the kind of resource a node shape targets. A form may uses them as the field label and help text.
- `rdfs:label` and `rdfs:comment` name and describe the shape resource itself. Note that for a `sh:ShapeClass`, these are `rdfs:label` and `rdfs:comment` properties of the class as well.
- `sh:intent` states in words a rule the shape is meant to express, including rules the shape cannot check.
- `sh:unit` gives the unit of measure or currency of the values, as a UCUM or ISO 4217 code or an IRI from a units vocabulary. It describes the values and but is not validated. 
- `sh:agentInstruction` is a text instruction for a software agent that reads the shape.
- `sh:codeIdentifier` is the name the shape gets in generated code, an API, or a table or column name. It is one string of letters, digits and underscores, not starting with a digit. `sh:codeType` is the datatype for the code to use for values.   Both are merely hints. SHACL does not convert values.
- `sh:order` is the general order of a shape among its siblings. It is often used ot sort property shapes and property groups in a form.


Below, a node shape and a property shape use these properties. 


```turtle
@prefix ex:   <http://example.org/> .
@prefix sh:   <http://www.w3.org/ns/shacl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

ex:PersonShape a sh:NodeShape ;
    sh:name "Person" ;                                  # the kind of resource the shape describes
    sh:description "A living individual that the organisation keeps records about." ;
    rdfs:label "Person shape" ;                         # the shape resource itself
    rdfs:comment "The shape that checks the data held about a person." ;
    sh:intent "A person's data is kept for as long as they are a customer." ;
    sh:codeIdentifier "customer" ;                      # the table, in a generated SQL schema
    sh:targetClass ex:Person ;
    sh:property ex:Person-weight .

ex:Person-weight a sh:PropertyShape ;
    sh:name "Weight" ;                                  # the property path, ex:weight
    sh:description "The person's body weight." ;
    sh:intent "Each person has a weight." {| sh:formalized true |} ;
    sh:intent "The weight is at least 20 kg." {| sh:formalized true |} ;
    sh:intent "The weight is measured, not estimated." ;   # a rule the shape cannot check
    sh:order 2 ;
    sh:unit "kg" ;
    sh:codeIdentifier "weight_kg" ;                     # the column
    sh:codeType xsd:decimal ;
    sh:agentInstruction "Do not estimate a weight that is missing. Ask for it." ;
    sh:path ex:weight ;
    sh:minCount 1 ;
    sh:datatype xsd:decimal ;
    sh:minInclusive 20 .
```